# Accessing OpenNeuro `ds000113` (StudyForrest), snapshot `1.3.0`

This notebook inspects the **pinned OpenNeuro snapshot** through its public GraphQL API, navigates the BIDS directory tree, downloads only files you choose, and opens NIfTI fMRI data lazily with NiBabel.

- Dataset: [OpenNeuro ds000113 v1.3.0](https://openneuro.org/datasets/ds000113/versions/1.3.0)
- DOI: [10.18112/openneuro.ds000113.v1.3.0](https://doi.org/10.18112/openneuro.ds000113.v1.3.0)
- BIDS version declared by the dataset: `1.1.0`

> **Storage warning:** this is a large, multimodal dataset. A single 7 T Forrest Gump BOLD run is often hundreds of MB, and each core participant has eight runs in both `acq-raw` and scanner distortion/motion-corrected `acq-dico` form. Start with one file.

## What is in this dataset?

Snapshot 1.3.0 combines four related historical OpenfMRI datasets, so its 36 numbered subject folders are **not one uniform 36-person experiment**.

| Part | Typical BIDS location | Content |
|---|---|---|
| Core StudyForrest cohort | `sub-01` …, `ses-forrestgump/` | 20 participants; eight ~15-minute runs of a German audio-description of *Forrest Gump*, 7 T, TR 2 s, 1.4 mm isotropic, partial-brain coverage; raw and scanner-corrected BOLD |
| Auditory perception | `ses-auditoryperception/func/` | 25 music clips in a slow event-related design, eight runs |
| Visual localizers | `ses-localizer/func/` | Movie localizer, object categories, and four retinotopic mapping tasks |
| 3 T movie | `ses-movie/func/` | Eight movie runs, event tables, and 1000 Hz eye gaze for an overlapping subset |
| Orientation-resolution study | `ses-r08`, `ses-r14`, `ses-r20`, `ses-r30` | Coverage and orientation tasks at 0.8, 1.4, 2, and 3 mm for seven participants |
| Structural/auxiliary | mostly `ses-forrestgump/{anat,dwi,fmap}` | T1w, T2w, venography, angiography, DWI, field maps, physiological traces, and defacing masks |
| Derived data | `derivatives/` | Motion estimates, templates, and linear/nonlinear anatomical alignments |
| Documentation/stimuli | `sourcedata/`, `stimuli/` | Acquisition protocols, DICOM information, stimulus annotations, and PsychoPy materials |
| Technical-noise scan | `sub-phantom/` | Phantom BOLD acquisitions |

Important details for analysis:

- `acq-raw` 7 T BOLD has severe geometric distortion.
- `acq-dico` has scanner-side distortion and motion correction and is generally the practical individual-subject starting point.
- The 7 T data omit superior brain regions; do not treat them as whole-brain acquisitions.
- Pre-aligned group-analysis images live under `derivatives/linear_anatomical_alignment/` and `derivatives/non-linear_anatomical_alignment/`.
- Many JSON sidecars are inherited from the dataset root under BIDS inheritance rules.

## 1. Setup

`urllib` is part of Python. Pandas is used for tables; NiBabel and Matplotlib are only needed when opening an image.

In [ ]:
# Run once if these packages are not already installed:
# %pip install -q pandas nibabel matplotlib


In [1]:
from pathlib import Path
from urllib.request import Request, urlopen
import io
import json

import pandas as pd

DATASET_ID = "ds000113"
SNAPSHOT = "1.3.0"
API_URL = "https://openneuro.org/crn/graphql"
DOWNLOAD_DIR = Path("openneuro_data") / f"{DATASET_ID}-{SNAPSHOT}"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("Downloads will go to:", DOWNLOAD_DIR.resolve())

Downloads will go to: /home/jovyan/junruz/forrestgump/openneuro_data/ds000113-1.3.0


## 2. Query the pinned snapshot

The API call below explicitly uses tag `1.3.0`. It does not silently switch to a newer draft or snapshot.

In [2]:
def graphql(query: str) -> dict:
    """Run a read-only query against OpenNeuro's public GraphQL API."""
    body = json.dumps({"query": query}).encode("utf-8")
    request = Request(
        API_URL,
        data=body,
        headers={"Content-Type": "application/json", "User-Agent": "ds000113-notebook/1.0"},
    )
    with urlopen(request, timeout=120) as response:
        result = json.load(response)
    if result.get("errors"):
        raise RuntimeError(result["errors"])
    return result["data"]


info_query = f'''query {{
  snapshot(datasetId: "{DATASET_ID}", tag: "{SNAPSHOT}") {{
    id
    tag
    description {{ Name BIDSVersion DatasetDOI Authors }}
  }}
}}'''

snapshot_info = graphql(info_query)["snapshot"]
snapshot_info

{'id': 'ds000113:1.3.0',
 'tag': '1.3.0',
 'description': {'Name': 'Forrest Gump',
  'BIDSVersion': '1.1.0',
  'DatasetDOI': '10.18112/openneuro.ds000113.v1.3.0',
  'Authors': ['Michael Hanke',
   'Florian J. Baumgartner',
   'Pierre Ibe',
   'Falko R. Kaule',
   'Stefan Pollmann',
   'Oliver Speck',
   'Wolf Zinke',
   'Jorg Stadler']}}

## 3. Navigate the exact BIDS tree

OpenNeuro represents directories as Git tree objects. The helper follows one directory at a time, which is much faster and lighter than asking the server for the entire large dataset tree.

In [3]:
_directory_cache = {}


def _files_for_tree(tree_id=None):
    tree_arg = "" if tree_id is None else f'(tree: {json.dumps(tree_id)})'
    query = f'''query {{
      snapshot(datasetId: "{DATASET_ID}", tag: "{SNAPSHOT}") {{
        files{tree_arg} {{ id filename size directory annexed urls }}
      }}
    }}'''
    return graphql(query)["snapshot"]["files"]


def list_directory(relative_dir=""):
    """List one directory in snapshot 1.3.0; paths are relative to its root."""
    relative_dir = relative_dir.strip("/")
    if relative_dir in _directory_cache:
        return _directory_cache[relative_dir]

    entries = _files_for_tree()
    traversed = []
    for part in filter(None, relative_dir.split("/")):
        traversed.append(part)
        cache_key = "/".join(traversed)
        match = next(
            (entry for entry in entries if entry["directory"] and entry["filename"] == part),
            None,
        )
        if match is None:
            raise FileNotFoundError(f"Directory not found in snapshot: {cache_key}")
        entries = _directory_cache.get(cache_key) or _files_for_tree(match["id"])
        _directory_cache[cache_key] = entries

    _directory_cache[relative_dir] = entries
    return entries


def as_table(entries):
    table = pd.DataFrame(entries).drop(columns="urls", errors="ignore")
    if "size" in table:
        table["size_MiB"] = pd.to_numeric(table["size"], errors="coerce") / 2**20
    return table.sort_values(["directory", "filename"], ascending=[False, True]).reset_index(drop=True)


root_entries = list_directory()
as_table(root_entries)

,id,filename,size,directory,annexed,size_MiB
0,f3dfb76e5040be01123e4e26eeeebecade8190b6,.datalad,0,True,False,0.000000
1,837417c36775302c587dbf6ff3fded9402c6c7f1,derivatives,0,True,False,0.000000
2,af82ebb8cc401d5ea9a60afb91baa81933555c14,sourcedata,0,True,False,0.000000
3,56566d84f6f4d04d8d73ef56d9d21088826e8ffb,stimuli,0,True,False,0.000000
4,8f18a9fbf8c2aca9d78c7185c8265a71fad47fd4,sub-01,0,True,False,0.000000
...,...,...,...,...,...,...
71,f411cb34abac8d130fa051e958b28f54f04df7e1,task-retmapclw_physio.json,253,False,False,0.000241
72,1cb2911930be270011e5e418694b98a16baa99a0,task-retmapcon_bold.json,997,False,False,0.000951
73,669499535b4648c6531178329423e204aa5836d7,task-retmapcon_physio.json,228,False,False,0.000217
74,574e976b48447f0f0efc59aa1dd8eb59b2275382,task-retmapexp_bold.json,996,False,False,0.000950


In [4]:
subject_folders = sorted(
    entry["filename"]
    for entry in root_entries
    if entry["directory"] and entry["filename"].startswith("sub-")
)
numbered_subjects = [s for s in subject_folders if s != "sub-phantom"]

print(f"Numbered subject folders: {len(numbered_subjects)}")
print("Other subject-like folders:", sorted(set(subject_folders) - set(numbered_subjects)))
print("First folders:", subject_folders[:10])

Numbered subject folders: 36
Other subject-like folders: ['sub-phantom']
First folders: ['sub-01', 'sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10']


The root `participants.tsv` contains session-level rows as well as participant fields. Inspect it before defining a cohort; folder count alone is not an experimental sample definition.

In [5]:
def file_entry(relative_path):
    """Resolve a file to its exact snapshot entry and API-provided URL(s)."""
    path = Path(relative_path)
    entries = list_directory("" if str(path.parent) == "." else path.parent.as_posix())
    match = next(
        (entry for entry in entries if not entry["directory"] and entry["filename"] == path.name),
        None,
    )
    if match is None:
        raise FileNotFoundError(f"File not found in {DATASET_ID} {SNAPSHOT}: {relative_path}")
    return match


def read_remote_bytes(relative_path):
    entry = file_entry(relative_path)
    if not entry.get("urls"):
        raise RuntimeError(f"OpenNeuro returned no download URL for {relative_path}")
    request = Request(entry["urls"][0], headers={"User-Agent": "ds000113-notebook/1.0"})
    with urlopen(request, timeout=120) as response:
        return response.read()


participants = pd.read_csv(io.BytesIO(read_remote_bytes("participants.tsv")), sep="\t")
print("Shape:", participants.shape)
display(participants.head())
display(participants.dtypes.to_frame("dtype"))

Shape: (37, 92)


,participant_id,gender,age,handedness,hearing_problems_current,hearing_problems_past,absolute_pitch,german_speak,german_comprehend,german_read,...,audioq1,audioq2,audioq3,audioq4,audioq5,audioq6,audioq7,audioq8,vision_problems_current,vision_problems_past
0,01,m,30-35,r,n,n,1.0,3.0,3.0,3.0,...,2.0,2.0,1.0,1.0,3.0,3.0,2.0,2.0,n,n
1,02,m,30-35,r,n,n,2.0,3.0,3.0,3.0,...,4.0,4.0,4.0,3.0,4.0,4.0,3.0,3.0,n,n
2,03,f,20-25,r,n,y,2.0,3.0,3.0,3.0,...,3.0,4.0,4.0,4.0,4.0,4.0,4.0,3.0,n,n
3,04,f,20-25,r,n,n,3.0,3.0,3.0,3.0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,n,n
4,05,m,25-30,r,n,n,3.0,3.0,3.0,3.0,...,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,n,n


,dtype
participant_id,object
gender,object
age,object
handedness,object
hearing_problems_current,object
...,...
audioq6,float64
audioq7,float64
audioq8,float64
vision_problems_current,object


## 4. Explore one participant/session

Change these values to the part of the dataset you need. Not every subject has every session.

In [6]:
SUBJECT = "sub-01"
SESSION = "ses-forrestgump"
MODALITY = "func"

print("Sessions available:")
display(as_table(list_directory(SUBJECT))[["filename", "directory"]])

selected_dir = f"{SUBJECT}/{SESSION}/{MODALITY}"
selected_entries = list_directory(selected_dir)
selected_table = as_table(selected_entries)
display(selected_table[["filename", "size_MiB", "annexed"]])

Sessions available:


,filename,directory
0,ses-auditoryperception,True
1,ses-forrestgump,True
2,ses-localizer,True
3,ses-movie,True


,filename,size_MiB,annexed
0,sub-01_ses-forrestgump_task-forrestgump_acq-di...,0.000873,False
1,sub-01_ses-forrestgump_task-forrestgump_acq-di...,445.335942,True
2,sub-01_ses-forrestgump_task-forrestgump_acq-di...,0.000873,False
3,sub-01_ses-forrestgump_task-forrestgump_acq-di...,435.581877,True
4,sub-01_ses-forrestgump_task-forrestgump_acq-di...,0.000873,False
5,sub-01_ses-forrestgump_task-forrestgump_acq-di...,431.636732,True
6,sub-01_ses-forrestgump_task-forrestgump_acq-di...,0.000873,False
7,sub-01_ses-forrestgump_task-forrestgump_acq-di...,481.414339,True
8,sub-01_ses-forrestgump_task-forrestgump_acq-di...,0.000873,False
9,sub-01_ses-forrestgump_task-forrestgump_acq-di...,454.737026,True


In [7]:
# Focus on scanner-corrected Forrest Gump BOLD images.
bold_candidates = selected_table[
    selected_table["filename"].str.contains("acq-dico", regex=False)
    & selected_table["filename"].str.endswith("_bold.nii.gz")
][["filename", "size_MiB"]]
display(bold_candidates)

,filename,size_MiB
1,sub-01_ses-forrestgump_task-forrestgump_acq-di...,445.335942
3,sub-01_ses-forrestgump_task-forrestgump_acq-di...,435.581877
5,sub-01_ses-forrestgump_task-forrestgump_acq-di...,431.636732
7,sub-01_ses-forrestgump_task-forrestgump_acq-di...,481.414339
9,sub-01_ses-forrestgump_task-forrestgump_acq-di...,454.737026
11,sub-01_ses-forrestgump_task-forrestgump_acq-di...,432.787206
13,sub-01_ses-forrestgump_task-forrestgump_acq-di...,534.964083
15,sub-01_ses-forrestgump_task-forrestgump_acq-di...,334.141891


## 5. Download only the selected file

The downloader resolves the file inside snapshot `1.3.0` and uses the URL returned for that exact file. Existing files are not overwritten unless `overwrite=True`.

In [ ]:
def download_file(relative_path, destination_root=DOWNLOAD_DIR, overwrite=False, chunk_mib=8):
    entry = file_entry(relative_path)
    if not entry.get("urls"):
        raise RuntimeError(f"OpenNeuro returned no download URL for {relative_path}")

    destination = Path(destination_root) / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and not overwrite:
        print("Already exists:", destination)
        return destination

    expected = int(entry["size"]) if entry.get("size") is not None else None
    request = Request(entry["urls"][0], headers={"User-Agent": "ds000113-notebook/1.0"})
    downloaded = 0
    with urlopen(request, timeout=300) as response, destination.open("wb") as output:
        while chunk := response.read(chunk_mib * 2**20):
            output.write(chunk)
            downloaded += len(chunk)
            if expected:
                print(f"\r{downloaded / 2**20:,.1f} / {expected / 2**20:,.1f} MiB", end="")
    print("\nSaved:", destination)
    if expected is not None and downloaded != expected:
        raise IOError(f"Size mismatch: downloaded {downloaded} bytes; expected {expected}")
    return destination


RELATIVE_BOLD = (
    "sub-01/ses-forrestgump/func/"
    "sub-01_ses-forrestgump_task-forrestgump_acq-dico_run-01_bold.nii.gz"
)

# This example is roughly 445 MiB. Uncomment when ready:
# bold_path = download_file(RELATIVE_BOLD)

Small metadata files use the same function. Dataset-level JSON sidecars such as `task-movie_bold.json` and `task-auditoryperception_bold.json` supply inherited metadata; run-specific JSON can add or override fields.

In [ ]:
dataset_description = json.loads(read_remote_bytes("dataset_description.json"))
movie_sidecar = json.loads(read_remote_bytes("task-movie_bold.json"))

display(dataset_description)
display(movie_sidecar)

## 6. Open and preview a downloaded NIfTI file

NiBabel initially reads only the header and keeps image data behind a proxy. The example loads the first 3D volume, not the entire 4D time series.

In [ ]:
# Run after the download cell above.
# import numpy as np
# import nibabel as nib
# import matplotlib.pyplot as plt
#
# img = nib.load(bold_path)
# print("Shape (x, y, z, time):", img.shape)
# print("Voxel sizes / TR:", img.header.get_zooms())
# print("Data type:", img.get_data_dtype())
#
# first_volume = np.asarray(img.dataobj[..., 0])
# z = first_volume.shape[2] // 2
# plt.figure(figsize=(7, 7))
# plt.imshow(first_volume[:, :, z].T, cmap="gray", origin="lower")
# plt.title(f"{SUBJECT}, first volume, axial slice z={z}")
# plt.axis("off");

## 7. Exact, reproducible access with DataLad (optional)

For a multi-file analysis, the dataset's Git/DataLad history is more convenient. Metadata are cloned first; large annexed objects are fetched only when requested. Checking out tag `1.3.0` pins the file tree. This requires `git-annex` and `datalad` on the system.

```bash
datalad clone https://github.com/OpenNeuroDatasets/ds000113.git ds000113
git -C ds000113 checkout 1.3.0
datalad -C ds000113 get sub-01/ses-forrestgump/func/sub-01_ses-forrestgump_task-forrestgump_acq-dico_run-01_bold.nii.gz
```

Useful selective patterns after cloning:

```bash
# One participant's scanner-corrected 7 T BOLD runs
datalad -C ds000113 get 'sub-01/ses-forrestgump/func/*acq-dico*_bold.nii.gz'

# One participant's T1w image
datalad -C ds000113 get sub-01/ses-forrestgump/anat/sub-01_ses-forrestgump_T1w.nii.gz

# Release local annexed content later while keeping the dataset/catalog
datalad -C ds000113 drop sub-01/ses-forrestgump/func/
```

`datalad drop` is intentionally shown only as a manual command: verify the path before running it.

## Suggested starting points

- **Individual-subject 7 T movie analysis:** begin with `ses-forrestgump`, `acq-dico`, one subject and one run.
- **Across-subject analysis:** consider the provided aligned derivatives, and read their reconstruction names carefully.
- **Event-related analysis:** `ses-auditoryperception`, `ses-localizer`, and `ses-movie` include event tables where applicable.
- **Naturalistic annotations:** inspect `stimuli/annotations/` for the audio-description transcript and scene boundaries.
- **Confound modeling:** retrieve the physiological recordings plus `derivatives/motion/`.

Always build the cohort from actual session/file availability rather than assuming every numbered subject took part in every acquisition.